# Charlie Eden

4/10/2026

In [1]:
# Setting working dir
import os
from pathlib import Path
p = os.getcwd() + "/../../"
os.chdir(p)
parent_dir = os.getcwd()

In [2]:
import numpy as np

config = {
    "years": np.arange(2019, 2025),
    "common_period": np.arange(2004, 2022),
    "imd_folder": f"{parent_dir}/../monsoon-benchmark_data/imd_rainfall_data/4p0",  # Ground truth rainfall data (4x4 degrees)
    "thresh_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",  # Threshold for the onset of the monsoon (4x4 degrees)
    "thres_file": f"{parent_dir}/../monsoon-benchmark_data/imd_onset_threshold/mwset4x4.nc4",  # Alternate naming convention
    "shpfile_path": f"{parent_dir}/../monsoon-benchmark_data/ind_map_shpfile/india_shapefile.shp",  # Shapefile of India
    "output_dir": f"{parent_dir}/examples/paper_figures/outputs",  # Directory to save data files,
    "mok": True,
    "day_bins_30": [(1, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)],
    "day_bins_15": [(1, 5), (6, 10), (11, 15)],
    "mem_num": 51,
    "date_filter_year": 2024,
    "file_pattern": "{}.nc",
    "data_dir": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0",
}

# Paths to 4p0 model forecast data (.nc)
model_paths = {
    "IFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/IFS_S2S",  # IFS model
    "AIFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/AIFS",  # AIFS model
    "FuXi": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi",  # FuXi mdoel
    "Graphcast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GraphCast",  # Graphcast model
    "GenCast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GenCast",  # GenCast model
    "FuXi-S2S": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S",  # FuXi_S2S model
    "NGCM": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/NeuralGCM",  # NGCM model
}

prob_model_paths = {
        "FuXi S2S": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi_S2S",  # FuXi_S2S model
        "NGCM": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/NeuralGCM",  # NGCM model
        "IFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/IFS_S2S",  # AIFS model
        "GenCast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GenCast",  # GenCast model
    }

det_model_paths = {
        "AIFS": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/AIFS",
        "FuXi": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/FuXi",
        "Graphcast": f"{parent_dir}/../monsoon-benchmark_data/rainfall_4p0/GraphCast",  # Graphcast model
    }

In [3]:
from monsoonbench.metrics import (
    ProbabilisticOnsetMetrics,
    ClimatologyOnsetMetrics,
    DeterministicOnsetMetrics
)
from monsoonbench.visualization.compare_models import calculate_reliability_metrics
import xarray as xr

c = ClimatologyOnsetMetrics()
p = ProbabilisticOnsetMetrics()
d = DeterministicOnsetMetrics()

skills_15 = {}
skills_30 = {}
brier_15 = {}
brier_30 = {}
auc_15 = {}
auc_30 = {}
reliability_15 = {}

thresh_ds = xr.open_dataset(config["thresh_file"])
thresh_slice = thresh_ds["MWmean"]
clim_onset = c.compute_climatological_onset_dataset(
            config["imd_folder"], thresh_slice, years=None, mok=config["mok"]
        )

Processing 124 years: [1901, 1902, 1903, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1913, 1914, 1915, 1916, 1917, 1918, 1919, 1920, 1921, 1922, 1923, 1924, 1925, 1926, 1927, 1928, 1929, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Processing year 1901...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd)

### Fig 2 -- Probabilistic Scores

In [4]:
# 15 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            range(2019, 2024),
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=15,
            day_bins=config["day_bins_15"],
            date_filter_year=config["date_filter_year"],
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["years"],
            config["day_bins_15"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=config["date_filter_year"],
            file_pattern=config["file_pattern"],
            max_forecast_day=15,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    reliability_metrics = calculate_reliability_metrics(forecast_df)

    skills_15[model_name] = skill_results
    reliability_15[model_name] = reliability_metrics
    auc_15[model_name] = auc_forecast
    brier_15[model_name] = brier_forecast
        



Processing IFS
Processing years: range(2019, 2024)
Using 4-degree CMZ polygon coordinates

Processing year 2019
Loading S2S model data...
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2019.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2019-06-02) as start date for onset detection
Found onset in 100 out of 100 grid points
Computing onset for all ensemble members...
Processing 26 init times x 10 unique locations x 11 members...
Unique lat-lon pairs: [(np.float64(76.0), np.float64(20.0)), (np.float64(80.0), np.float64(20.0)), (np.float64(84.0), np.float64(20.0)), (np.float64(72.0), np.float64(24.0)), (np.float64(76.0), np.float64(24.0)), (np.float64(80.0), np.float64(24.0)), (np.float64(84.0), np.float64(24.0)), (np.float64(72.0), np.float64(28.0)), (np.float64(76.0), np.float64(28.0)), (np.float64(80.0), np.float64(28.

KeyboardInterrupt: 

In [ ]:
# 30 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            range(2019, 2024),
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=30,
            day_bins=config["day_bins_30"],
            date_filter_year=config["date_filter_year"],
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["years"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=config["date_filter_year"],
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    skills_30[model_name] = skill_results  
    auc_30[model_name] = auc_forecast
    brier_30[model_name] = brier_forecast      

In [ ]:
import pandas as pd
rows = []
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 15

    row["fair_brier_skill_d1_5"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = None
    row["fair_brier_skill_d21_25"] = None
    row["fair_brier_skill_d26_30"] = None

    row["fair_brier_d1_5"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = brier_15[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = None
    row["fair_brier_d21_25"] = None
    row["fair_brier_d26_30"] = None

    row["auc_d1_5"] = auc_15[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = auc_15[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = auc_15[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = None
    row["auc_d21_25"] = None
    row["auc_d26_30"] = None
    row["auc_later"] = auc_15[model_name]["bin_auc_scores"]["After day 15"]

    rows.append(row)


    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 30

    row["fair_brier_skill_d1_5"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 16-20"]
    row["fair_brier_skill_d21_25"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 21-25"]
    row["fair_brier_skill_d26_30"] = skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 26-30"]

    row["fair_brier_d1_5"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 16-20"]
    row["fair_brier_d21_25"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 21-25"]
    row["fair_brier_d26_30"] = brier_30[model_name]["bin_fair_brier_scores"]["Days 26-30"]

    row["auc_d1_5"] = auc_30[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = auc_30[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = auc_30[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = auc_30[model_name]["bin_auc_scores"]["Days 16-20"]
    row["auc_d21_25"] = auc_30[model_name]["bin_auc_scores"]["Days 21-25"]
    row["auc_d26_30"] = auc_30[model_name]["bin_auc_scores"]["Days 26-30"]
    row["auc_later"] = auc_30[model_name]["bin_auc_scores"]["After day 30"]

    rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df


In [ ]:
metrics_df.to_csv(f"{config["output_dir"]}/fig2_data_recreation.csv")

In [ ]:
reliability_15

### Fig 3 -- Binned Avg. CMZ Spatial Metrics

In [ ]:
# Reading in Matlab files

from scipy.io import loadmat

def load_mat_to_dict(file_path:str,
                    vars_of_interest: list = None):
    data = loadmat(file_path)
    if vars_of_interest:
        ret_dict = {var: data[var] for var in vars_of_interest}
        return ret_dict
    return data

In [ ]:
load_mat_to_dict("/Users/charlieeden/Downloads/5day_forecastwindow_cmz_2019_2024.mat")

In [ ]:
from monsoonbench.utils.forecast_window_pipeline import (
    run_multi_model_window_analysis,
    get_default_four_degree_model_specs
)

model_specs = {
        "ifs": {
            "model_type": "probabilistic",
            "expected_ens": 11,
            "mem_num": 11,
            "date_filter_year": 2024,
            "path": model_paths["IFS"],
        },
        "aifs": {
            "model_type": "deterministic",
            "expected_ens": None,
            "mem_num": None,
            "date_filter_year": 2024,
            "path": model_paths["AIFS"],
        },
        "fuxi": {
            "model_type": "deterministic",
            "expected_ens": None,
            "mem_num": None,
            "date_filter_year": 2024,
            "path": model_paths["FuXi"],
        },
        "graphcast": {
            "model_type": "deterministic",
            "expected_ens": None,
            "mem_num": None,
            "date_filter_year": 2024,
            "path": model_paths["Graphcast"],
        },
        "gencast": {
            "model_type": "probabilistic",
            "expected_ens": 51,
            "mem_num": 51,
            "date_filter_year": 2024,
            "path": model_paths["GenCast"],
        },
        "fuxi-s2s": {
            "model_type": "probabilistic",
            "expected_ens": 51,
            "mem_num": 51,
            "date_filter_year": 2022,
            "path": model_paths["FuXi-S2S"],
        },
        "ngcm": {
            "model_type": "probabilistic",
            "expected_ens": 51,
            "mem_num": 51,
            "date_filter_year": 2024,
            "path": model_paths["NGCM"],
        },
    }

fig3_spatial_cmz_data = run_multi_model_window_analysis(base_config=config,
                                model_specs=model_specs,
                                )

In [ ]:
for key, value in fig3_spatial_cmz_data["matrices"].items():
    value.to_csv(
        f"{config["output_dir"]}/fig3_data_recreation_{key}.csv"
    )
    print(f"CSV for {key} saved")

In [ ]:
common_skills_15 = {}
common_skills_30 = {}
common_brier_15 = {}
common_brier_30 = {}
common_auc_15 = {}
common_auc_30 = {}
common_reliability_15 = {}

In [ ]:
# 15 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            config["common_period"],
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=15,
            day_bins=config["day_bins_15"],
            date_filter_year=2022,
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_15"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=15,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    reliability_metrics = calculate_reliability_metrics(forecast_df)

    common_skills_15[model_name] = skill_results
    common_reliability_15[model_name] = reliability_metrics
    common_auc_15[model_name] = auc_forecast
    common_brier_15[model_name] = brier_forecast

# 30 day
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    print("\n")
    print(f"Processing {model_name}")
    print("="*50)
    forecast_df = p.multi_year_forecast_obs_pairs(
            config['common_period'],
            model_forecast_dir=model_paths[model_name],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            mem_num=51,
            max_forecast_day=30,
            day_bins=config["day_bins_30"],
            date_filter_year=2022,
        )
    
    climatology_obs_df = c.multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )
    
    brier_forecast = p.calculate_brier_score(forecast_df)
    rps_forecast = p.calculate_rps(forecast_df)
    auc_forecast = p.calculate_auc(forecast_df)

    brier_climatology = c.calculate_brier_score_climatology(climatology_obs_df)
    rps_climatology = p.calculate_rps(climatology_obs_df)
    auc_climatology = c.calculate_auc_climatology(climatology_obs_df)


    skill_results = p.calculate_skill_scores(
            brier_forecast, rps_forecast, brier_climatology, rps_climatology
        )
    
    common_skills_30[model_name] = skill_results  
    common_auc_30[model_name] = auc_forecast
    common_brier_30[model_name] = brier_forecast      
        

In [ ]:
rows = []
for model_name in [
    "IFS",
    "GenCast",
    "FuXi-S2S",
    "NGCM",
    ]:
    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 15

    row["fair_brier_skill_d1_5"] = common_skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = common_skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = common_skills_15[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = None
    row["fair_brier_skill_d21_25"] = None
    row["fair_brier_skill_d26_30"] = None

    row["fair_brier_d1_5"] = common_brier_15[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = common_brier_15[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = common_brier_15[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = None
    row["fair_brier_d21_25"] = None
    row["fair_brier_d26_30"] = None

    row["auc_d1_5"] = common_auc_15[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = common_auc_15[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = common_auc_15[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = None
    row["auc_d21_25"] = None
    row["auc_d26_30"] = None
    row["auc_later"] = common_auc_15[model_name]["bin_auc_scores"]["After day 15"]

    rows.append(row)


    row = {}
    row["model_label"] = model_name.lower()
    row["horizon"] = 30

    row["fair_brier_skill_d1_5"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 1-5"]
    row["fair_brier_skill_d6_10"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 6-10"]
    row["fair_brier_skill_d11_15"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 11-15"]
    row["fair_brier_skill_d16_20"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 16-20"]
    row["fair_brier_skill_d21_25"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 21-25"]
    row["fair_brier_skill_d26_30"] = common_skills_30[model_name]["bin_fair_brier_skill_scores"]["Days 26-30"]

    row["fair_brier_d1_5"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 1-5"]
    row["fair_brier_d6_10"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 6-10"]
    row["fair_brier_d11_15"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 11-15"]
    row["fair_brier_d16_20"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 16-20"]
    row["fair_brier_d21_25"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 21-25"]
    row["fair_brier_d26_30"] = common_brier_30[model_name]["bin_fair_brier_scores"]["Days 26-30"]

    row["auc_d1_5"] = common_auc_30[model_name]["bin_auc_scores"]["Days 1-5"]
    row["auc_d6_10"] = common_auc_30[model_name]["bin_auc_scores"]["Days 6-10"]
    row["auc_d11_15"] = common_auc_30[model_name]["bin_auc_scores"]["Days 11-15"]
    row["auc_d16_20"] = common_auc_30[model_name]["bin_auc_scores"]["Days 16-20"]
    row["auc_d21_25"] = common_auc_30[model_name]["bin_auc_scores"]["Days 21-25"]
    row["auc_d26_30"] = common_auc_30[model_name]["bin_auc_scores"]["Days 26-30"]
    row["auc_later"] = common_auc_30[model_name]["bin_auc_scores"]["After day 30"]

    rows.append(row)

common_metrics_df = pd.DataFrame(rows)
common_metrics_df


In [ ]:
common_metrics_df.to_csv(
    f"{config["output_dir"]}/fig3_common_period_data_recreation.csv"
)

### Fig 6 -- Probabilistic Metrics 4x4 lon/lat

In [4]:
brier_model_paths = {
        "FuXi-S2S": model_paths["FuXi-S2S"],
        "NGCM": model_paths["NGCM"],
        "IFS": model_paths["IFS"],
    }

In [5]:
from examples.paper_figures.fig6_data_loader import (
    fig6_multi_year_forecast_obs_pairs,
    fig6_multi_year_climatological_forecast_obs_pairs
)

forecast_dfs_15 = {}
forecast_dfs_30 = {}
for model_name, model_fp in brier_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    multi_year_df = fig6_multi_year_forecast_obs_pairs(
        range(2004, 2022),
        model_forecast_dir=model_fp,
        imd_folder=config["imd_folder"],
        thres_file=config["thresh_file"],
        mem_num=51 if model_name != "IFS" else 11,
        max_forecast_day=15,
        day_bins=config["day_bins_15"],
        date_filter_year=2024 if model_name != "IFS" else 2022,
    )
    forecast_dfs_15[model_name] = multi_year_df

    multi_year_df = fig6_multi_year_forecast_obs_pairs(
        range(2004, 2022),
        model_forecast_dir=model_fp,
        imd_folder=config["imd_folder"],
        thres_file=config["thresh_file"],
        mem_num=51 if model_name != "IFS" else 11,
        max_forecast_day=30,
        day_bins=config["day_bins_30"],
        date_filter_year=2024 if model_name != "IFS" else 2022,
    )
    forecast_dfs_30[model_name] = multi_year_df

# brier_15_ds = {}
# brier_30_ds = {}
# rps_15_ds = {}
# rps_30_ds = {}

# for model, df in forecast_dfs_15.items():
#     loop_grid_brier = calculate_gridwise_brier(df)
#     brier_15_ds[model] = loop_grid_brier

# for model, df in forecast_dfs_30.items():
#     loop_grid_brier = calculate_gridwise_brier(df)
#     brier_30_ds[model] = loop_grid_brier

# for model, df in forecast_dfs_15.items():
#     loop_grid_rps = calculate_gridwise_rps(df)
#     rps_15_ds[model] = loop_grid_rps

# for model, df in forecast_dfs_30.items():
#     loop_grid_rps = calculate_gridwise_rps(df)
#     rps_30_ds[model] = loop_grid_rps

# brier_rps_graph_data_list = [brier_15_ds, brier_30_ds, rps_15_ds, rps_30_ds]

Loading data from FuXi-S2S
Processing years: range(2004, 2022)
Using 4-degree CMZ polygon coordinates
Lat Check
[ 8. 12. 16. 20. 24. 28. 32. 36.]

Processing year 2004
Loading S2S model data...
ERROR CHECK [ 68.  72.  76.  80.  84.  88.  92.  96. 100.]:
Loading IMD rainfall data...
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/2004.nc
Renamed dimensions: {'TIME': 'time'}
Detecting observed onset...
Using MOK date (June 2nd) (2004-06-02) as start date for onset detection
Found onset in 36 out of 72 grid points
Computing onset for all ensemble members...
p_model_slice coords: [ 8. 12. 16. 20. 24. 28. 32. 36.] [ 68.  72.  76.  80.  84.  88.  92.  96. 100.]
onset_da coords: [ 8. 12. 16. 20. 24. 28. 32. 36.] [ 68.  72.  76.  80.  84.  88.  92.  96. 100.]
onset_da non-NaN count: 36
Processing 13 init times x 72 unique locations x 51 members...
Using MOK (June 2nd filter) for onset detection
Processing

In [7]:
# brier climatology
climatology_obs_df_15 = fig6_multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_15"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=15,
            mok=config["mok"],
        )

climatology_obs_df_30 = fig6_multi_year_climatological_forecast_obs_pairs(
            clim_onset,
            config["common_period"],
            config["day_bins_30"],
            config["mem_num"],
            model_paths[model_name],
            date_filter_year=2022,
            file_pattern=config["file_pattern"],
            max_forecast_day=30,
            mok=config["mok"],
        )

Using 4-degree CMZ polygon coordinates

Processing target year 2004
Creating climatological forecasts for target year 2004
Using 124 years as ensemble members: [np.int64(1901), np.int64(1902), np.int64(1903), np.int64(1904), np.int64(1905), np.int64(1906), np.int64(1907), np.int64(1908), np.int64(1909), np.int64(1910), np.int64(1911), np.int64(1912), np.int64(1913), np.int64(1914), np.int64(1915), np.int64(1916), np.int64(1917), np.int64(1918), np.int64(1919), np.int64(1920), np.int64(1921), np.int64(1922), np.int64(1923), np.int64(1924), np.int64(1925), np.int64(1926), np.int64(1927), np.int64(1928), np.int64(1929), np.int64(1930), np.int64(1931), np.int64(1932), np.int64(1933), np.int64(1934), np.int64(1935), np.int64(1936), np.int64(1937), np.int64(1938), np.int64(1939), np.int64(1940), np.int64(1941), np.int64(1942), np.int64(1943), np.int64(1944), np.int64(1945), np.int64(1946), np.int64(1947), np.int64(1948), np.int64(1949), np.int64(1950), np.int64(1951), np.int64(1952), np.int6

In [8]:
clim_brier_15 = c.calculate_brier_score_climatology(climatology_obs_df_15)
clim_rps_15 = p.calculate_rps(climatology_obs_df_15)

clim_brier_30 = c.calculate_brier_score_climatology(climatology_obs_df_30)
clim_rps_30 = p.calculate_rps(climatology_obs_df_30)

Calculating Brier Score excluding 'Before initialization' bin
Original samples: 22310, After filtering: 17848
Brier Score (excluding 'Before initialization'): 0.0839
Fair Brier Score (excluding 'Before initialization'): 0.0834
Bins included in calculation: ['After day 15', 'Days 1-5', 'Days 11-15', 'Days 6-10']
RPS: 0.4097
Fair RPS: 0.4074
Number of forecasts: 4462
Calculating Brier Score excluding 'Before initialization' bin
Original samples: 35696, After filtering: 31234
Brier Score (excluding 'Before initialization'): 0.0825
Fair Brier Score (excluding 'Before initialization'): 0.0820
Bins included in calculation: ['After day 30', 'Days 1-5', 'Days 11-15', 'Days 16-20', 'Days 21-25', 'Days 26-30', 'Days 6-10']
RPS: 0.8204
Fair RPS: 0.8156
Number of forecasts: 4462


In [9]:
import pandas as pd

In [15]:
def fig6_metric_calculation(forecast_df,
                            n=15,
                            model_name=None):
    rows= []
    print(len(forecast_df.lat.unique()))
    for lat in forecast_df.lat.unique():
        for lon in forecast_df.lon.unique():
            print("="*50)
            print(f"Calculating for {lat}, {lon} pair")
            print("="*50)
            row = {}
            loop_df = forecast_df.loc[
                (forecast_df["lat"] == lat) & (forecast_df["lon"] == lon)
                ].copy()
            if loop_df.empty:
                print(f"Skipping {lat}, {lon} — no data")
                continue
            else:
                brier = p.calculate_brier_score(loop_df)
                rps = p.calculate_rps(loop_df)
                skill_scores = p.calculate_skill_scores(
                            brier_forecast=brier,
                            rps_forecast=rps,
                            brier_climatology=clim_brier_15 if n==15 else clim_brier_30,
                            rps_climatology=clim_rps_15 if n==15 else clim_rps_30
                        )
                row["BSS"] = skill_scores["fair_brier_skill_score"]
                row["RPS"] = skill_scores["fair_rps_skill_score"]
                row["lat"] = lat
                row["lon"] = lon
                row["horizon"] = n
                if model_name:
                    row["model"] = model_name
                rows.append(row)

    return pd.DataFrame(rows)
                        

fig6_metrics_dict_15 = {}
fig6_metrics_dict_30 = {}
for key, value in forecast_dfs_15.items():
    loop_ret = fig6_metric_calculation(value, n=15, model_name=key)
    fig6_metrics_dict_15[key] = loop_ret

for key, value in forecast_dfs_30.items():
    loop_ret = fig6_metric_calculation(value, n=30, model_name=key)
    fig6_metrics_dict_30[key] = loop_ret
    

8
Calculating for 8.0, 76.0 pair
Brier Score: 0.1288
Fair Brier Score: 0.1286
RPS: 0.3637
Fair RPS: 0.3632
Number of forecasts: 99
SKILL SCORE CALCULATIONS
Fair Brier Skill Score (1-15 day): -0.5421
Fair RPS Skill Score (1-15 day): 0.1086

Automatically detected target bins: ['Days 1-5', 'Days 6-10', 'Days 11-15']
Excluded bins: ['After day 15']

Bin-wise Fair Brier Skill Scores:
  Days 1-5: Fair BSS = -0.1795
  Days 6-10: Fair BSS = -2.2152
  Days 11-15: Fair BSS = -1.2412

SKILL SCORE SUMMARY TABLE
Metric                         Overall (1-15 day) 1-5          6-10         11-15       
------------------------------------------------------------------------------------
Fair Brier Skill Score         -0.5421            -0.1795      -2.2152      -1.2412     
Fair RPS Skill Score           0.1086             N/A          N/A          N/A         
------------------------------------------------------------------------------------

Interpretation Guide:
• Positive skill scores indicate f

In [20]:
fig6_metrics_15 = pd.concat(fig6_metrics_dict_15.values())
fig6_metrics_30 = pd.concat(fig6_metrics_dict_30.values())

fig6_metrics = pd.concat([
    fig6_metrics_15,
    fig6_metrics_30
])

In [22]:
fig6_metrics.to_csv(f"{config["output_dir"]}/fig6_data_recreation.csv")

### Fig. 7 - 12 -- Spatial Metrics 4x4 lon/lat grid

In [23]:
model_years = {
    "FuXi S2S": [2019, 2020, 2021],
    "IFS": [2019, 2020, 2021, 2022, 2023],
    "Standard": [2019, 2020, 2021, 2022, 2023, 2024],
}

In [24]:
# 15 day
metrics_df_clim_15, onset_da_clim_15 = (
        c.compute_climatology_baseline_multiple_years(
            years=model_years["Standard"],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            tolerance_days=3,
            verification_window=1,
            forecast_days=15,
            max_forecast_day=15,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

spatial_clim_15_day = c.create_spatial_far_mr_mae(
        metrics_df_clim_15, dict.fromkeys(model_years["Standard"], onset_da_clim_15)
    )

model_dfs_15 = {}
model_onsets_15 = {}

for model_name, model_fp in prob_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    probabilistic_df_15, onset_da_dict_15 = (
        p.compute_metrics_multiple_years(
            years=(
                model_years[model_name]
                if model_name in model_years.keys()
                else model_years["Standard"]
            ),
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            model_forecast_dir=model_fp,
            tolerance_days=3,
            verification_window=1,
            forecast_days=15,
            max_forecast_day=15,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

    model_dfs_15[model_name] = probabilistic_df_15
    model_onsets_15[model_name] = onset_da_dict_15

for model_name, model_fp in det_model_paths.items():
        print("=" * 80)
        print(f"Loading data from {model_name}")
        print("=" * 80)
        deterministic_df_15, onset_da_dict_15 = (
            d.compute_metrics_multiple_years(
                years=(
                    model_years[model_name]
                    if model_name in model_years.keys()
                    else model_years["Standard"]
                ),
                imd_folder=config["imd_folder"],
                thres_file=config["thresh_file"],
                model_forecast_dir=model_fp,
                tolerance_days=3,
                verification_window=1,
                forecast_days=15,
                max_forecast_day=15,
                mok=True,
                onset_window=5,
                mok_month=6,
                mok_day=2,
            )
        )

        model_dfs_15[model_name] = deterministic_df_15
        model_onsets_15[model_name] = onset_da_dict_15

Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_da

In [25]:
# 30 Day
metrics_df_clim_30, onset_da_clim_30 = (
        c.compute_climatology_baseline_multiple_years(
            years=model_years["Standard"],
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            tolerance_days=5,
            verification_window=16,
            forecast_days=30,
            max_forecast_day=30,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

spatial_clim_30_day = c.create_spatial_far_mr_mae(
        metrics_df_clim_30, dict.fromkeys(model_years["Standard"], onset_da_clim_30)
    )

model_dfs_30 = {}
model_onsets_30 = {}

for model_name, model_fp in prob_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    probabilistic_df_30, onset_da_dict_30 = (
        p.compute_metrics_multiple_years(
            years=(
                model_years[model_name]
                if model_name in model_years.keys()
                else model_years["Standard"]
            ),
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            model_forecast_dir=model_fp,
            tolerance_days=5,
            verification_window=16,
            forecast_days=30,
            max_forecast_day=30,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )
    model_dfs_30[model_name] = probabilistic_df_30
    model_onsets_30[model_name] = onset_da_dict_30

for model_name, model_fp in det_model_paths.items():
    print("=" * 80)
    print(f"Loading data from {model_name}")
    print("=" * 80)
    deterministic_df_30, onset_da_dict_30 = (
        d.compute_metrics_multiple_years(
            years=(
                model_years[model_name]
                if model_name in model_years.keys()
                else model_years["Standard"]
            ),
            imd_folder=config["imd_folder"],
            thres_file=config["thresh_file"],
            model_forecast_dir=model_fp,
            tolerance_days=5,
            verification_window=16,
            forecast_days=30,
            max_forecast_day=30,
            mok=True,
            onset_window=5,
            mok_month=6,
            mok_day=2,
        )
    )

    model_dfs_30[model_name] = deterministic_df_30
    model_onsets_30[model_name] = onset_da_dict_30


Computing climatological onset reference...
Computing climatological onset from 124 years: 1901-2024
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1901.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1901-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1902.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1902-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_data/4p0/1903.nc
Renamed dimensions: {'TIME': 'time'}
Using MOK date (June 2nd) (1903-06-02) as start date for onset detection
Loading IMD rainfall from: /Users/charlieeden/Desktop/Spring 2026/ds_clinic/monsoon-bench/../monsoon-benchmark_data/imd_rainfall_da

In [51]:
def format_data_for_spatial_fig(
        tuple_of_model_data_dicts: tuple, clim_data: xr.DataArray
) -> dict:
    """Format and merge model and climatology metrics into a single ordered dict for plotting.

    Computes spatial FAR/MR/MAE metrics for each probabilistic and deterministic model,
    scales rates to percentages, appends the climatology baseline, and reorders all
    entries to match the paper's figure layout.

    Parameters
    ----------
    parent_dir : str
        Root directory used to resolve model path configurations.
    tuple_of_model_data_dicts : tuple
        Two-element tuple of (model_dfs, model_onsets) as returned by
        ``load_model_data_fig_7_through_9`` or ``load_model_data_fig_10_through_12``.
    clim_data : xr.DataArray
        Climatology spatial metrics DataArray as returned by the corresponding
        ``load_clim_data_*`` function.

    Returns:
    -------
    dict
        Ordered dictionary mapping model name to spatial metrics xr.Dataset,
        with keys in paper figure order:
        ['Climatology', 'IFS', 'AIFS', 'FuXi', 'Graphcast', 'GenCast', 'FuXi S2S', 'NGCM'].
    """

    def reorder_dict(dict) -> dict:
        """Reorders dict to match paper format."""
        order = [
            "Climatology",
            "IFS",
            "AIFS",
            "FuXi",
            "Graphcast",
            "GenCast",
            "FuXi S2S",
            "NGCM",
        ]
        reordered_dict = {key: dict[key] for key in order}
        return reordered_dict

    prob_plot_data = {}

    model_dfs = tuple_of_model_data_dicts[0]
    model_onsets = tuple_of_model_data_dicts[1]

    for model_name in prob_model_paths.keys():
        probabilistic_df = model_dfs[model_name]
        onset_da_dict = model_onsets[model_name]
        plot_probabilistic_metrics = p.create_spatial_far_mr_mae(
            probabilistic_df, onset_da_dict
        )
        plot_probabilistic_metrics["false_alarm_rate"] = (
            plot_probabilistic_metrics["false_alarm_rate"].round(3) * 100
        )
        plot_probabilistic_metrics["miss_rate"] = (
            plot_probabilistic_metrics["miss_rate"].round(3) * 100
        )
        prob_plot_data[model_name] = plot_probabilistic_metrics

    for model_name in det_model_paths.keys():
        deterministic_df = model_dfs[model_name]
        onset_da_dict = model_onsets[model_name]
        plot_deterministic_metrics = d.create_spatial_far_mr_mae(
            deterministic_df, onset_da_dict
        )
        plot_deterministic_metrics["false_alarm_rate"] = (
            plot_deterministic_metrics["false_alarm_rate"].round(3) * 100
        )
        plot_deterministic_metrics["miss_rate"] = (
            plot_deterministic_metrics["miss_rate"].round(3) * 100
        )

        prob_plot_data[model_name] = plot_deterministic_metrics

    clim_data["false_alarm_rate"] = clim_data["false_alarm_rate"].round(3) * 100
    clim_data["miss_rate"] = clim_data["miss_rate"].round(3) * 100

    prob_plot_data["Climatology"] = clim_data

    prob_plot_data = reorder_dict(prob_plot_data)

    return prob_plot_data

In [52]:
spatial_figs_15 = format_data_for_spatial_fig(
    tuple_of_model_data_dicts=(model_dfs_15, model_onsets_15),
    clim_data=spatial_clim_15_day
)

Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]


In [ ]:
spatial_figs_30 = format_data_for_spatial_fig(
    tuple_of_model_data_dicts=(model_dfs_30, model_onsets_30),
    clim_data=spatial_clim_30_day
)

Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]
Creating spatial FAR, Miss Rate, yearly MAE, and mean MAE maps...
Grid dimensions: 8 lats x 9 lons
Years: [2019, 2020, 2021, 2022, 2023, 2024]


In [76]:
def make_df_long(df_short, metric):
    df_long = (
    df_short
    .stack()                # collapse columns into rows
    .reset_index()          # turn index into columns
    .rename(columns={
        "level_0": "lat",
        "level_1": "lon",
        0: metric
    })
)
    return df_long

spatial_dfs_final = []
for key, value in spatial_figs_15.items():
    mae_df = value["mean_mae"].to_pandas()
    mae_df_long = make_df_long(mae_df, "Mean MAE")
    mae_df_long["model"] = key
    mae_df_long["horizon"] = 15

    mr_df = value["miss_rate"].to_pandas()
    mr_df_long = make_df_long(mr_df, "Miss Rate")
    mr_df_long["model"] = key
    mr_df_long["horizon"] = 15


    far_df = value["false_alarm_rate"].to_pandas()
    far_df_long = make_df_long(far_df, "False Alarm Rate")
    far_df_long["model"] = key
    far_df_long["horizon"] = 15

    merged_df = (
    mae_df_long
    .merge(mr_df_long, on=["lat", "lon", "model", "horizon"], how="outer")
    .merge(far_df_long, on=["lat", "lon", "model", "horizon"], how="outer")
    )

    spatial_dfs_final.append(merged_df)

for key, value in spatial_figs_30.items():
    mae_df = value["mean_mae"].to_pandas()
    mae_df_long = make_df_long(mae_df, "Mean MAE")
    mae_df_long["model"] = key
    mae_df_long["horizon"] = 30

    mr_df = value["miss_rate"].to_pandas()
    mr_df_long = make_df_long(mr_df, "Miss Rate")
    mr_df_long["model"] = key
    mr_df_long["horizon"] = 30


    far_df = value["false_alarm_rate"].to_pandas()
    far_df_long = make_df_long(far_df, "False Alarm Rate")
    far_df_long["model"] = key
    far_df_long["horizon"] = 30

    merged_df = (
    mae_df_long
    .merge(mr_df_long, on=["lat", "lon", "model", "horizon"], how="outer")
    .merge(far_df_long, on=["lat", "lon", "model", "horizon"], how="outer")
    )

    spatial_dfs_final.append(merged_df)

In [78]:
spatial_final_csv = pd.concat(spatial_dfs_final)
spatial_final_csv.to_csv(f"{config["output_dir"]}/fig_7_through_12_data_recreation.csv")